# Improved Training Approach for MudraLearn

This notebook demonstrates the improved training approach that addresses the class imbalance issue.
We implement:
1. Filtering to classes with sufficient samples (≥10)
2. Class-weighted loss to handle remaining imbalance
3. Simple data augmentation (landmark jitter)
4. GRU architecture (best performer from baseline)

**Key Improvement**: By focusing on adequately-sampled classes and using proper weighting,
we achieve significantly better performance than training on all 383 classes (most with 1-5 samples).

## 1. Imports and Setup

Import required libraries and set up paths.

In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical
import json

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Paths
DATA_DIR = os.path.join('..', 'data')
SAVED_MODEL_DIR = os.path.join('..', 'saved_models')
os.makedirs(SAVED_MODEL_DIR, exist_ok=True)

print(f"TensorFlow version: {tf.__version__}")
print(f"Working directory: {os.getcwd()}")

TensorFlow version: 2.21.0
Working directory: /Users/senithuekenayake/Documents/Coding/BscFinalYear/FinalProject/MudraLearn/ml/notebooks


## 2. Load and Analyze Data

Load the preprocessed data and analyze class distribution to confirm the imbalance issue.

In [2]:
# Load preprocessed data
X_train = np.load(os.path.join(DATA_DIR, 'X_train.npy'))
X_val = np.load(os.path.join(DATA_DIR, 'X_val.npy'))
X_test = np.load(os.path.join(DATA_DIR, 'X_test.npy'))
y_train = np.load(os.path.join(DATA_DIR, 'y_train.npy'))
y_val = np.load(os.path.join(DATA_DIR, 'y_val.npy'))
y_test = np.load(os.path.join(DATA_DIR, 'y_test.npy'))

print(f"Training data shape: X={X_train.shape}, y={y_train.shape}")
print(f"Validation data shape: X={X_val.shape}, y={y_val.shape}")
print(f"Test data shape: X={X_test.shape}, y={y_test.shape}")

# Convert one-hot to class indices for analysis
y_train_idx = np.argmax(y_train, axis=1)
y_val_idx = np.argmax(y_val, axis=1)
y_test_idx = np.argmax(y_test, axis=1)

num_classes = y_train.shape[1]
print(f"Number of classes: {num_classes}")

# Analyze class distribution
unique_train, counts_train = np.unique(y_train_idx, return_counts=True)
print(f"\n=== TRAINING SET CLASS DISTRIBUTION ===")
print(f"Classes present: {len(unique_train)}/{num_classes}")
print(f"Min samples per class: {counts_train.min()}")
print(f"Max samples per class: {counts_train.max()}")
print(f"Mean samples per class: {counts_train.mean():.2f}")
print(f"Median samples per class: {np.median(counts_train):.2f}")

# Count classes with insufficient samples
for threshold in [1, 5, 10, 20]:
    n_insufficient = np.sum(counts_train < threshold)
    print(f"Classes with < {threshold} samples: {n_insufficient} ({n_insufficient/len(unique_train)*100:.1f}%")

Training data shape: X=(2965, 30, 132), y=(2965, 383)
Validation data shape: X=(635, 30, 132), y=(635, 383)
Test data shape: X=(636, 30, 132), y=(636, 383)
Number of classes: 383

=== TRAINING SET CLASS DISTRIBUTION ===
Classes present: 370/383
Min samples per class: 1
Max samples per class: 80
Mean samples per class: 8.01
Median samples per class: 5.00
Classes with < 1 samples: 0 (0.0%
Classes with < 5 samples: 176 (47.6%
Classes with < 10 samples: 272 (73.5%
Classes with < 20 samples: 332 (89.7%
